# 第6回 収益率と変動の大きさ


第5回の演習1の追加問題では、レートが80円前後だった2012年と150円前後だった2024年について、1日の動きの大きさを円で比べた。80円のときの1円の動きと150円のときの1円の動きを、同じ大きさの動きとして比べてよいのだろうか。

この回ではレートの動きを前の日の値に対する割合で表し、ドルを持つ人の収益率として読む。収益率がどのくらい散らばるかを標準偏差で測り、これを変動の大きさと呼ぶ。1日の変動の大きさと1年の変動の大きさの関係を調べ、最後に、1年後に旅行する人にとってレートが1年でどのくらい動くことがあるかを考えよう。


## 実行の準備

`finance_data.zip` をNotebookと同じフォルダに置き、下のセルを実行する。セルは上から順に実行すること。「変数が見つからない」というエラーが出たら、前のセルを実行し忘れている。


In [ ]:
from pathlib import Path
from zipfile import ZipFile

if not Path("course_data.py").exists():
    with ZipFile("finance_data.zip") as archive:
        archive.extractall(".")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from course_data import (
    load_boj_fx, load_boj_monthly, load_fed_fx, load_fed_rates,
    configure_plotting, format_axis,
)

configure_plotting()
pd.options.display.float_format = "{:.3f}".format


## 第5回 演習1の解説

第5回の演習1から、問3と追加問題を振り返ろう。まず第5回と同じ手順で、東京17時のレート `rate` と、今日の値を次の日の予想にしたときの表 `forecast` を作り直す。


In [ ]:
boj = load_boj_fx()
rate = boj["usd_jpy"].dropna()
forecast = pd.DataFrame({
    "予想（前日の値）": rate.shift(1),
    "当日の値": rate,
})
forecast["誤差"] = forecast["当日の値"] - forecast["予想（前日の値）"]
error_2024 = forecast.loc["2024"]["誤差"]
print(f"今日の値による予想の誤差の大きさの平均（2024年）：{error_2024.abs().mean():.2f} 円／ドル")


### 問3：過去の平均による予想

`rate.loc[:"2023"]` で2023年までのすべての行を取り出し、`.mean()` で平均を求める。この平均を2024年のすべての日の予想にするので、誤差は2024年のレートから平均を引いた差になる。


In [ ]:
past_mean = rate.loc[:"2023"].mean()
mean_error = rate.loc["2024"] - past_mean
print(f"2000〜2023年の平均：{past_mean:.1f} 円／ドル")
print(f"平均による予想の誤差の大きさの平均（2024年）：{mean_error.abs().mean():.2f} 円／ドル")
print(f"今日の値による予想の誤差との比：{mean_error.abs().mean() / error_2024.abs().mean():.0f} 倍")


平均による予想の誤差の大きさの平均は42.42円で、今日の値を使う予想の0.70円の約61倍になった。2024年のレートと平均の109.2円の位置を図で比べてみよう。


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(rate.loc[:"2024"].index, rate.loc[:"2024"], linewidth=0.8)
ax.axhline(past_mean, color="gray", linewidth=0.8)
ax.set(title="ドル・円（東京17時）と2000〜2023年の平均", ylabel="円／ドル")
format_axis(ax)
fig.tight_layout()
plt.show()


灰色の線が2000〜2023年の平均109.2円である。2024年のレートは1年を通して140円を上回り、平均の線から30円以上離れていた。第5回で見たように当日の値は前日の値の近くにあるので、次の日の予想には、遠い過去まで含めた平均より今日の値を使うほうがよい。

2024年のレートの平均を予想にした人もいるだろう。誤差の大きさの平均は42.42円よりずっと小さくなるが、2024年の平均は2024年が終わるまで分からない値である。予想を作るときは、予想する時点で分かっていた値だけを使おう。


### 追加問題：2012年と2024年の比較

2012年の誤差の大きさの平均は、2024年と同じく `forecast.loc["2012"]` から求める。倍率は2012年の値を2024年の値で割って求める。


In [ ]:
size_2012 = forecast.loc["2012"]["誤差"].abs().mean()
size_2024 = error_2024.abs().mean()
mean_2012 = rate.loc["2012"].mean()
mean_2024 = rate.loc["2024"].mean()
print(f"誤差の大きさの平均：2012年 {size_2012:.2f}、2024年 {size_2024:.2f} 円／ドル（{size_2012 / size_2024:.2f} 倍）")
print(f"レートの平均：2012年 {mean_2012:.1f}、2024年 {mean_2024:.1f} 円／ドル（{mean_2012 / mean_2024:.2f} 倍）")


2012年の誤差の大きさの平均は0.28円で、2024年の0.70円の0.41倍だった。レートの平均は79.8円と151.6円で、こちらは0.53倍である。誤差がレートの水準に比例するなら誤差の倍率も0.53倍ほどになるはずだが、実際の0.41倍はそれより小さい。2012年の誤差が小さかったのは、水準が低かったことに加えて、レートの動きそのものが穏やかだったからである。

表示された0.28と0.70で割った人は0.40倍になったはずで、丸める前の値で割ると0.41倍になる。2012年の動きがどれだけ穏やかだったかは、この回で水準の違いを除いて確かめる。


## 80円のときの1円と150円のときの1円

10万円をドルに両替した2人を考えよう。Aさんはレートが80円のときに、Bさんは150円のときに両替した。次の日にどちらのレートも1円上がった（1円の円安になった）とする。2人が持っているドルを円に戻すと、10万円からいくら増えるかを計算してみる。


In [ ]:
yen = 100000
for price in [80, 150]:
    dollars = yen / price
    gain = dollars * (price + 1) - yen
    print(f"{price}円で両替：{dollars:.2f} ドル、1円の円安で {gain:.0f} 円増える（{100 * gain / yen:.2f}%）")


同じ1円の円安で、Aさんは1250円、Bさんは667円増えた。もとの10万円に対する割合は1.25%と0.67%である。1円は80円の1.25%、150円の0.67%にあたるので、レートが低いときの1円の動きほど、ドルを持つ人の損得の割合は大きくなる。水準の違う日の動きを比べるには、前の日の値に対する割合を使おう。


## 変化率

第1回では、株価の変化率を「（あとの価格 − はじめの価格）÷ はじめの価格 × 100」で求めた。はじめの価格を前日の値、あとの価格を当日の値にすれば、1日の変化率になる。単位は%である。

$$\text{変化率} = \frac{\text{当日の値} - \text{前日の値}}{\text{前日の値}} \times 100$$

ドルを持っている人にとって、変化率は円で数えた金額が1日で何%増えたか（減ったか）を表す。持っている資産から得た収益の割合を収益率と呼び、ドル・円の変化率はドルを持つことの収益率にあたる。

解説のセルで作り直した `rate` から、第5回と同じく `rate.diff()` で前日からの変化を求め、前日の値 `rate.shift(1)` で割る。


In [ ]:
change = rate.diff()
pct = 100 * change / rate.shift(1)

pd.DataFrame({
    "当日の値": rate,
    "前日の値": rate.shift(1),
    "変化（円）": change,
    "変化率（%）": pct,
}).loc["2024-01-04":"2024-01-12"]


1月4日は前日（2023年12月29日）の141.40円から1.98円の円安で、変化率は 1.98 ÷ 141.40 × 100 = 1.40% になる。1月9日は1.00円の円高で、変化率は −0.69% だった。変化率の符号は変化と同じで、プラスなら円安、マイナスなら円高を表す。

26年分の変化と変化率を、上下に並べた図で比べよう。`plt.subplots(2, 1)` は図の枠を縦に2つ作り、上の枠を `axes[0]`、下の枠を `axes[1]` で指定する。`sharex=True` を付けると2つの図の横軸がそろう。


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
axes[0].plot(change.index, change, linewidth=0.5)
axes[0].set(title="前日からの変化（東京17時）", ylabel="円／ドル")
axes[1].plot(pct.index, pct, linewidth=0.5)
axes[1].set(title="変化率（東京17時）", ylabel="%")
for ax in axes:
    ax.axhline(0, color="gray", linewidth=0.8)
    format_axis(ax)
fig.tight_layout()
plt.show()


上の円の図では、2022年以降の振れ幅が2011〜2012年よりずっと大きい。下の変化率の図にすると、2つの時期の振れ幅の違いは小さくなる。2022年以降のレートは130〜160円ほどで、80円前後だった2011〜2012年よりずっと高く、同じ割合の動きでも円で測った幅は大きくなるからである。

第5回の追加問題で比べた2012年と2024年について、変化の大きさの平均と変化率の大きさの平均を並べよう。どちらも `.abs()` で符号を外してから平均する。


In [ ]:
for year in ["2012", "2024"]:
    size_yen = change.loc[year].abs().mean()
    size_pct = pct.loc[year].abs().mean()
    print(f"{year}年：変化の大きさの平均 {size_yen:.2f} 円／ドル、変化率の大きさの平均 {size_pct:.2f}%")

ratio = pct.loc["2012"].abs().mean() / pct.loc["2024"].abs().mean()
print(f"変化率の大きさの平均の比（2012年 ÷ 2024年）：{ratio:.2f} 倍")


円で測ると2012年の0.28円は2024年の0.70円の0.41倍だった。変化率で測ると0.35%と0.46%で、2つの年の差は縮まり、比は0.77倍になる。それでも2012年の値は2024年より小さく、水準の違いを除いても2012年の動きは穏やかだった。

円で測った0.41倍は、レートの水準の比0.53倍に変化率の比0.77倍を掛けた値（0.53 × 0.77 = 0.41）にほぼ等しい。円で測った動きの違いには、レートの水準の違いと、割合で見た動きの違いの2つが含まれている。


## 日ごとの変化率から1年の収益率へ

2023年の最後の観測日（12月29日）にドルを買い、2024年の最後の観測日（12月30日）まで持っていた人の1年の収益率を、日ごとの変化率から求めてみよう。

ある日の変化率が1.40%なら、その日にドルの円での価値は1.0140倍になる。1年の価値の変化は、この倍率を1年分の日数だけ掛け合わせれば求まる。`.prod()` は列の値をすべて掛け合わせる関数である。比べるために、日ごとの変化率を `.sum()` でそのまま足し合わせた値も表示する。`.sum()` は列の値をすべて足し合わせる関数である。


In [ ]:
start = rate.loc["2023-12-29"]
end = rate.loc["2024-12-30"]
growth = (1 + pct.loc["2024"] / 100).prod()
print(f"2023年12月29日：{start:.2f} 円／ドル、2024年12月30日：{end:.2f} 円／ドル")
print(f"最初と最後の値から求めた1年の変化率：{100 * (end - start) / start:.2f}%")
print(f"日ごとの倍率を掛け合わせて求めた変化率：{100 * (growth - 1):.2f}%")
print(f"日ごとの変化率をそのまま足した値：{pct.loc['2024'].sum():.2f}%")


日ごとの倍率を掛け合わせた値は、最初と最後の値から求めた11.66%と一致した。1年の収益率を求めるだけなら掛け算で足りる。日ごとの変化率をそのまま足した11.58%は、少しずれている。

ただし変化率には、上げと下げで大きさがそろわないという性質がある。レートが80円から150円になると変化率は +87.5% だが、150円から80円に戻ると −46.7% で、同じ道のりを往復したのに大きさが違う。円安の日と円高の日を同じ物差しで比べ、平均や標準偏差で散らばりを測るには、上げと下げが同じ大きさになる測り方がほしい。それが次に説明する対数変化率である。


## 対数変化率

対数には、掛け算を足し算に直す性質がある。

$$\log(a \times b) = \log a + \log b$$

経済データ分析Ⅰで対数目盛のグラフを見た人は、同じ間隔が同じ倍率を表していたことを思い出そう。これも、倍率の掛け算が対数の足し算になる性質による。

日ごとの倍率（当日の値 ÷ 前日の値）の対数に100を掛けた値を対数変化率という。日ごとの倍率を掛け合わせる計算は、対数変化率を足し合わせる計算に置き換わる。金融のデータの分析では、この対数変化率を収益率として使うことが多い。

$$\text{対数変化率} = \log\frac{\text{当日の値}}{\text{前日の値}} \times 100 = (\log \text{当日の値} - \log \text{前日の値}) \times 100$$

$\log$ は自然対数を表し、Pythonでは `np.log()` で計算する。`np.log(rate)` はレートの各行の対数を並べた列で、その `.diff()` が前日との対数の差になる。まず変化率と並べてみよう。


In [ ]:
log_pct = 100 * np.log(rate).diff()

pd.DataFrame({
    "変化率（%）": pct,
    "対数変化率（%）": log_pct,
}).loc["2024-01-04":"2024-01-12"]


1月4日の変化率1.400%に対して、対数変化率は1.391%である。ほかの日も2つの差は0.01%以下で、1日の動きならどちらで測ってもほぼ同じ値になる。

動きが大きいと2つの差は開く。前の節の80円と150円の往復で比べてみよう。


In [ ]:
for before, after in [(80, 150), (150, 80)]:
    print(f"{before}円→{after}円：変化率 {100 * (after / before - 1):+.1f}%、対数変化率 {100 * np.log(after / before):+.1f}%")


変化率では +87.5% と −46.7% だった往復が、対数変化率では +62.9% と −62.9% になった。対数変化率なら上げと下げが同じ大きさになり、円安の日と円高の日を同じ物差しで比べられる。

次に、2024年の対数変化率を足し合わせ、1年の対数変化率と比べよう。足し合わせた値は `np.exp()` で倍率に戻せる。`np.exp()` は `np.log()` の逆の計算をする関数である。


In [ ]:
log_total = log_pct.loc["2024"].sum()
print(f"1年の対数変化率：{100 * (np.log(end) - np.log(start)):.2f}%")
print(f"2024年の対数変化率の合計：{log_total:.2f}%")
print(f"合計を倍率に戻して求めた変化率：{100 * (np.exp(log_total / 100) - 1):.2f}%")


対数変化率を足し合わせた値は、1年の対数変化率11.03%と一致した。倍率に戻すと11.66%で、日ごとの倍率を掛け合わせた値と同じになる。対数変化率を足す計算は、倍率を掛け合わせる計算と同じ答えを出す。1年の対数変化率（11.03%）が変化率（11.66%）より小さいのは、1年で1割を超えるような大きな動きでは2つの差が開くからである。


## ドルを1年持っていたときの収益率

対数変化率を年ごとに足し合わせれば、前の年の最後の観測日にドルを買い、その年の最後の観測日まで持っていたときの収益率が求まる。

年ごとに足すには `.groupby()` を使う。`log_pct.groupby(log_pct.index.year)` は、日付の年（`.index.year`）が同じ行をまとめる。その後に `.sum()` を付けると、まとめた行ごとに合計を計算する。2000年は前の年の最後の値がなく1年分の収益率が求まらないので、`.loc[2001:]` で2001年以降を使う。


In [ ]:
annual = log_pct.groupby(log_pct.index.year).sum().loc[2001:]
annual.loc[2021:2025]


左の列が年、右の列がその年にドルを持っていたときの収益率（%）である。2024年の11.031%は、前のセルで求めた値と同じになっている。2021〜2024年はプラスで、ドルを持っていた人は円で数えた金額が増えた。2025年は −1.2%で、わずかに減った。

25年分を棒グラフにしよう。`ax.bar()` に横軸の位置（年）と棒の高さ（収益率）を渡すと、棒グラフが描ける。


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(annual.index, annual)
ax.axhline(0, color="gray", linewidth=0.8)
ax.set(title="ドルを1年持っていたときの収益率（対数変化率、東京17時）", ylabel="%")
format_axis(ax, dates=False)
fig.tight_layout()
plt.show()


棒が上に伸びた年は円安で、ドルを持っていた人が得をした年である。下に伸びた年は円高で、損をした年にあたる。平均と、最も大きく増えた年と減った年を確かめておく。


In [ ]:
print(f"25年の平均：{annual.mean():.1f}%")
print(f"最も大きく増えた年：{annual.idxmax()} 年（{annual.max():.1f}%）")
print(f"最も大きく減った年：{annual.idxmin()} 年（{annual.min():.1f}%）")


平均すると1年に1.2%の円安だったが、2013年には19.9%増え、2008年には22.6%減った。10万円分のドルを持っていたなら、2013年は約2万円増え、2008年は約2万円減った計算になる。1年の収益率は、平均の1.2%よりずっと広い幅で散らばっている。この散らばりの大きさを1つの数で表すのが、次に説明する標準偏差である。


## 標準偏差

標準偏差は、値が平均のまわりにどれだけ散らばっているかを表す数で、次の手順で求める。

1. 各日の対数変化率から、その期間の平均を引く。
2. 引いた差を2乗する。マイナスの差も2乗すればプラスになる。
3. 2乗した差を足し合わせ、日数から1を引いた数で割る。この値を分散という。
4. 分散の平方根をとる。2乗した値を元の単位（%）に戻すためである。

3で日数から1を引いた数で割るのは統計で決まった計算の仕方で、日数が多ければ日数で割った値とほとんど変わらない。

金融のデータの分析では、収益率の標準偏差を変動の大きさ（ボラティリティ）と呼ぶ。まず1日の変動の大きさを測る。2024年について、手順を1つずつ計算しよう。`** 2` は2乗、`np.sqrt()` は平方根を求める書き方である。


In [ ]:
g_2024 = log_pct.loc["2024"]
deviation = g_2024 - g_2024.mean()
squared = deviation ** 2
variance = squared.sum() / (len(g_2024) - 1)
print(f"日数：{len(g_2024)} 日、平均：{g_2024.mean():.3f}%")
print(f"分散（2乗した差の合計 ÷ {len(g_2024) - 1}）：{variance:.3f}")
print(f"標準偏差：{np.sqrt(variance):.3f}%")


同じ計算は `.std()` を使えば1行で書ける。`std` は英語の standard deviation（標準偏差）からとった名前である。2000〜2025年の26年分の標準偏差も求めておく。


In [ ]:
print(f"2024年の標準偏差：{g_2024.std():.3f}%")
sd_day = log_pct.std()
print(f"2000〜2025年の標準偏差：{sd_day:.3f}%")


2024年は手順どおりに求めた値と同じ0.668%になった。2024年の1日の変動の大きさは0.67%ほどで、レートが150円なら1円ほどにあたる（150 × 0.67 ÷ 100 = 1.0）。26年分では0.619%である。この値を、26年を通した1日の変動の大きさとして使う。


## 年ごとの標準偏差

1日の変動の大きさは年によって変わる。年ごとの収益率と同じく `.groupby()` で年ごとにまとめ、`.sum()` のかわりに `.std()` を付けて、各年の1日の標準偏差を求めよう。


In [ ]:
sd_year = log_pct.groupby(log_pct.index.year).std()
print(f"2012年：{sd_year.loc[2012]:.3f}%、2024年：{sd_year.loc[2024]:.3f}%")
print(f"最も大きい年：{sd_year.idxmax()} 年（{sd_year.max():.3f}%）")
print(f"最も小さい年：{sd_year.idxmin()} 年（{sd_year.min():.3f}%）")


2012年は0.462%で、2024年の0.668%の0.69倍だった。水準の違いを除いて比べても、2012年は2024年より穏やかな年だったことが分かる。最も大きいのは2008年の1.024%、最も小さいのは2021年の0.341%で、3倍ほどの開きがある。2008年はアメリカで始まった金融危機が世界に広がった年で、収益率の棒グラフでも −22.6% と最も大きく減った年だった。

同じ計算を円の変化 `change` でも行い、レートの水準との関係を2つの散布図で比べよう。横軸は各年のレートの平均、縦軸は各年の標準偏差で、1つの点が1年を表す。`plt.subplots(1, 2)` は図の枠を横に2つ並べる。第5回で使った `.corr()` で相関係数も求める。


In [ ]:
sd_yen_year = change.groupby(change.index.year).std()
mean_year = rate.groupby(rate.index.year).mean()

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].plot(mean_year, sd_yen_year, "o")
axes[0].set(title="変化（円）の標準偏差", xlabel="年平均のレート（円／ドル）", ylabel="円／ドル")
axes[1].plot(mean_year, sd_year, "o")
axes[1].set(title="対数変化率の標準偏差", xlabel="年平均のレート（円／ドル）", ylabel="%")
for ax in axes:
    format_axis(ax, dates=False)
fig.tight_layout()
plt.show()

print(f"レートの平均と変化（円）の標準偏差の相関係数：{mean_year.corr(sd_yen_year):.2f}")
print(f"レートの平均と対数変化率の標準偏差の相関係数：{mean_year.corr(sd_year):.2f}")


左の円の図ではレートの平均が高い年ほど標準偏差も大きく、相関係数は0.56である。右の対数変化率の図では右上がりの並びが見えなくなり、相関係数は −0.03 になった。円で測った変動の大きさは、レートの水準にほぼ比例して大きくなる。対数変化率で測れば水準の影響が除かれ、水準の違う年の変動の大きさを同じ物差しで比べられる。


## 1日の変動の大きさと1年の変動の大きさ

1日の変動の大きさは26年分で0.619%だった。1年の変動の大きさは、その何倍になるだろうか。2024年の観測日は245日だったので、0.619%の245倍になるだろうか。

「ドルを1年持っていたときの収益率」の節で求めた `annual` は、各年の245日ほどの対数変化率を足し合わせた値だった。その25年分の標準偏差が、1年の変動の大きさにあたる。1日の標準偏差を245倍した値と並べよう。


In [ ]:
print(f"1年ごとの収益率の標準偏差（2001〜2025年）：{annual.std():.2f}%")
print(f"1日の標準偏差の245倍：{sd_day * 245:.1f}%")
print(f"1日の標準偏差の√245倍：{sd_day * np.sqrt(245):.2f}%")


245倍した151.6%は、実際の10.09%とまるで合わない。1年の変動の大きさが151.6%もあるなら、レートが1年で何倍にもなる年が珍しくないことになるが、26年の間でさえレートは76円から162円の間で動いただけである。一方、√245倍（約15.7倍）した9.69%は実際の値に近い。

1年の変動の大きさが1日の245倍にならないのは、毎日の動きの向きがそろわないからである。第5回で見たように、前日の変化と当日の変化の相関係数は −0.019 で、前日の向きから次の日の向きは読めなかった。円安の日と円高の日が打ち消し合うので、245日分を足しても1日の動きの245倍にはならない。

毎日の動きどうしに相関がなく、1日の変動の大きさが期間を通して同じなら、n 日分を足した対数変化率の分散は1日の分散の n 倍になる。標準偏差は分散の平方根なので、次の関係が成り立つ。

$$\text{n 日の標準偏差} = \text{1日の標準偏差} \times \sqrt{n}$$

期間が長くなるほど変動の大きさは大きくなる。ただし増え方は期間の長さの平方根に比例し、期間の長さに比例するよりずっとゆるやかである。


1年より短い期間でも確かめよう。`np.log(rate).shift(-n)` は、n 観測日あとの対数を同じ行に入れる。第4回で `rate.shift(-1)` を使って次の日の値を同じ行に入れたのと同じ書き方である。そこから今日の対数を引けば、各日から n 観測日あとまでの対数変化率になる。1日、5日（1週間）、21日（1か月）、245日（1年）について、その標準偏差を1日の標準偏差の √n 倍と n 倍に並べる。


In [ ]:
log_rate = np.log(rate)
for n in [1, 5, 21, 245]:
    move = 100 * (log_rate.shift(-n) - log_rate)
    print(f"{n} 日分：実際 {move.std():.2f}%、√{n} 倍 {sd_day * np.sqrt(n):.2f}%、{n} 倍 {sd_day * n:.1f}%")


どの期間でも、実際の値は √n 倍の値とほぼ同じで、n 倍の値とは大きく離れている。21日（1か月）分の対数変化率の標準偏差は2.81%で、√21倍の2.84%とほぼ一致した。

5日から245日まで5日おきに実際の値を求め、√n 倍の曲線と重ねて描こう。`.append()` は、リストの最後に値を1つ付け加える書き方である。


In [ ]:
days = list(range(5, 250, 5))
actual_sd = []
for n in days:
    actual_sd.append((100 * (log_rate.shift(-n) - log_rate)).std())

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(days, actual_sd, "o", markersize=4, label="実際の値")
ax.plot(days, sd_day * np.sqrt(days), color="gray", label="1日の標準偏差の √n 倍")
ax.set(title="n 日分の対数変化率の標準偏差", xlabel="n（観測日）", ylabel="%")
ax.legend()
format_axis(ax, dates=False)
fig.tight_layout()
plt.show()


点は灰色の曲線に沿って並んでいる。曲線ははじめ急に上がり、n が大きくなるほど上がり方がゆるやかになる。曲線の上で25日分は 0.619 × √25 = 3.1% で、その4倍の100日分は 0.619 × √100 = 6.2% と2倍にとどまる。


## 旅行の予算と1年の変動の大きさ

1年後に海外旅行をする人が、今のレートで1000ドル分の旅費を見積もるとしよう。1年の間に、レートはどのくらい動くことがあるだろうか。

1年の変動の大きさを1年ごとの収益率から直接求めると、値は1年に1つしか得られない。2001〜2025年の25年分を集めても標準偏差を計算する値は25個で、その中には変動の大きさがまったく違う年が混ざる。毎日のデータなら2025年の1年だけで243個ある。2025年の1日の変動の大きさを √n 倍すれば、2025年と同じくらいの変動が続いたときの n 日の変動の大きさが求まる。

2025年の最後の観測日（12月30日）のレートで1000ドルの旅費を計算し、1か月（21日）、3か月（63日）、1年（245日）の変動の大きさを円の金額に直そう。


In [ ]:
last_rate = rate.loc["2025-12-30"]
budget = 1000 * last_rate
sd_2025 = log_pct.loc["2025"].std()
print(f"2025年12月30日のレート：{last_rate:.2f} 円／ドル、1000ドルは {budget:,.0f} 円")
print(f"2025年の1日の変動の大きさ（標準偏差）：{sd_2025:.3f}%（{log_pct.loc['2025'].count()} 日）")
for n in [21, 63, 245]:
    sd_n = sd_2025 * np.sqrt(n)
    print(f"{n} 日の変動の大きさ：{sd_n:.2f}%（1000ドルの旅費の約 {round(budget * sd_n / 100, -2):,.0f} 円）")


2025年末のレートで1000ドルの旅費は155,980円である。2025年の1日の変動の大きさ0.587%から求めると、1年の変動の大きさは9.19%で、旅費の約14,300円にあたる。1か月なら約4,200円、3か月なら約7,300円で、両替までの期間が長いほど大きくなる。

標準偏差は散らばりの目安で、1年の動きがその中に必ず収まるという意味ではない。過去の1年ごとの収益率が −9.19% から +9.19% の間に何年収まったかを数えておく。


In [ ]:
sd_1y = sd_2025 * np.sqrt(245)
print(f"−{sd_1y:.2f}% から +{sd_1y:.2f}% の間：{(annual.abs() <= sd_1y).sum()} 年")
print(f"+{sd_1y:.2f}% を超えた円安：{(annual > sd_1y).sum()} 年")
print(f"−{sd_1y:.2f}% を下回った円高：{(annual < -sd_1y).sum()} 年")


2001〜2025年の25年のうち、−9.19% から +9.19% の間に収まったのは13年だった。残りの12年、半分ほどの年はそれより大きく動いている。9.19%を超える円安になった8年には、今のレートより1割ほど高く見積もった旅費でも足りなかったことになる。

この計算は、毎日の変動の大きさが2025年と同じくらい続くと考えたものである。年ごとの標準偏差は0.341%から1.024%まで変わったので、変動の大きい年が来れば、1年の変動の大きさも9.19%より大きくなる。


## 演習1：変化率による比較

1. 問1の計算欄のコードは、2000〜2025年の東京17時の値について、変化（円）の大きさが大きかった5日と、対数変化率の大きさが大きかった5日を表に表示する。セルを実行し、2つの表の両方に入っている日と、対数変化率の表だけに入っている日を答えよ。また、対数変化率の表だけに入っている日のレート（当日の値）は、変化（円）の表だけに入っている日のレートより高いか低いかを答えよ。
2. 次の(a)〜(c)は正しいか。正しいか誤りかを答え、それぞれ理由を1文で書け。
   - (a) レートが100円から101円になった日と、150円から151円になった日の変化率は同じである。
   - (b) 1日の対数変化率の標準偏差が0.6%なら、21日分の対数変化率の標準偏差は 0.6 × 21 = 12.6% ほどになる。
   - (c) 年ごとの標準偏差を変化（円）で求めると、レートの平均が高い年ほど大きくなりやすい。
3. 問3の計算欄のコードは、FRBのニューヨーク正午の値から、ドル・円（円／ドル）とユーロ・ドル（ドル／ユーロ）の年ごとの標準偏差を2つの図に描く。左の図は前日からの変化をそれぞれの単位のまま使い、右の図は対数変化率を使っている。ドル・円とユーロ・ドルのどちらが大きく動いたといえるかを、2つの図の縦軸の単位に注意して3〜5文で説明せよ。

追加問題：2024年の対数変化率について、月ごとの標準偏差を求めよ。標準偏差が最も大きい月と最も小さい月を答え、最も大きい月の標準偏差が2024年全体の標準偏差（0.668%）の何倍かを求めよ。

問1と問3は計算欄のセルを実行して表示された結果を読み、問2は本文の式と図をもとに答える。追加問題では、本文のセルを参考にしてコードを書く。


In [ ]:
# 問1の計算欄：変化（円）と対数変化率で、大きく動いた5日を比べる
big_yen = change.abs().nlargest(5).index
big_log = log_pct.abs().nlargest(5).index
days = pd.DataFrame({
    "当日の値": rate,
    "変化（円）": change,
    "対数変化率（%）": log_pct,
})
print("変化（円）の大きさが大きかった5日")
print(days.loc[big_yen])
print()
print("対数変化率の大きさが大きかった5日")
print(days.loc[big_log])


In [ ]:
# 問3の計算欄：ドル・円とユーロ・ドルの年ごとの標準偏差
fed = load_fed_fx()
pairs = {
    "ドル・円（円／ドル）": fed["usd_jpy"].dropna(),
    "ユーロ・ドル（ドル／ユーロ）": fed["eur_usd"].dropna(),
}

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for name, series in pairs.items():
    raw_sd = series.diff().groupby(series.index.year).std()
    log_sd = (100 * np.log(series).diff()).groupby(series.index.year).std()
    axes[0].plot(raw_sd.index, raw_sd, marker="o", markersize=3, label=name)
    axes[1].plot(log_sd.index, log_sd, marker="o", markersize=3, label=name)
    print(f"{name}：対数変化率の標準偏差（2000〜2025年）{(100 * np.log(series).diff()).std():.3f}%")
axes[0].set(title="変化の標準偏差（それぞれの単位）", ylabel="円／ドル、ドル／ユーロ")
axes[1].set(title="対数変化率の標準偏差", ylabel="%")
for ax in axes:
    format_axis(ax, dates=False)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=2)
fig.tight_layout(rect=(0, 0.08, 1, 1))
plt.show()


In [ ]:
# 追加問題の計算欄
# 2024年の対数変化率は g_2024 に入っている。


### 演習1の記入欄

- 問1：両方の表に入っている日、対数変化率の表だけに入っている日、そのレートの比較：
- 問2(a)：正しいか誤りか、その理由：
- 問2(b)：正しいか誤りか、その理由：
- 問2(c)：正しいか誤りか、その理由：
- 問3（3〜5文）：
- 追加問題：最も大きい月と最も小さい月、2024年全体の何倍か：


<details>
<summary>演習1のヒント</summary>

問1では、2つの表の日付を見比べよう。片方の表だけに入っている日について当日の値の列を見ると、レートが高い時期の日か低い時期の日かが分かる。

問2の(a)は、「変化率」の節の式に数を当てはめて計算する。(b)は「1日の変動の大きさと1年の変動の大きさ」の節の式と、21日分の実際の値を見よう。(c)は「年ごとの標準偏差」の節の左の散布図と相関係数を見よう。

問3では、ドル・円とユーロ・ドルの値がそれぞれどのくらいの大きさかを思い出そう。ドル・円は1ドルが100円を超え、ユーロ・ドルは1ユーロが1ドル前後である。値そのものが小さければ、1日の変化も小さな数になる。説明には、右の図や表示された26年分の標準偏差を使ってよい。

追加問題の月ごとの標準偏差は、「年ごとの標準偏差」の節のセルで `log_pct` を `g_2024` に、`.index.year` を `.index.month` に替えると求められる。2024年全体の標準偏差は `g_2024.std()` である。

</details>


## 出典

[日本銀行「外国為替市況」](https://www.boj.or.jp/statistics/market/forex/fxdaily/index.htm)の東京市場ドル・円スポット、17時時点（円／ドル）を使用。配布データの対象期間は2000〜2025年である。

演習1の問3で使う米国の為替レート（ドル・円、ユーロ・ドル）は [FRB H.10（為替）](https://www.federalreserve.gov/releases/h10/)による。
